# Train LightGBM on Merged Dataset

This notebook:
1. Trains a LightGBM model on the merged dataset (80% Dataset B + Dataset C)
2. Tests the model on Training_For_Meta_Model.csv (20% of Dataset B)

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# Load the datasets
train_data = pd.read_csv('solo_gbm_dataset/Dataset_B_80_plus_C.csv')
test_data = pd.read_csv('Meta Model Dataset/Training_For_Meta_Model.csv')

print(f"Training data shape: {train_data.shape}")
print(f"Test data shape: {test_data.shape}")

Training data shape: (56137, 11)
Test data shape: (13652, 12)


In [4]:
# Display column info
print("\nTraining data columns:")
print(train_data.columns.tolist())
print("\nTest data columns:")
print(test_data.columns.tolist())

print("\nTraining data info:")
print(train_data.info())


Training data columns:
['Age', 'Gender', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Cholesterol Level', 'Glucose Level', 'Smoking Status', 'Alcohol Intake', 'Physical Activity', 'Cardiovascular Disease', 'BMI']

Test data columns:
['Age', 'Alcohol Intake', 'BMI', 'Cardiovascular Disease', 'Cholesterol Level', 'Diastolic Blood Pressure', 'Gender', 'Glucose Level', 'Physical Activity', 'Smoking Status', 'Systolic Blood Pressure', 'id']

Training data info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56137 entries, 0 to 56136
Data columns (total 11 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       56137 non-null  float64
 1   Gender                    56137 non-null  int64  
 2   Systolic Blood Pressure   56137 non-null  float64
 3   Diastolic Blood Pressure  56137 non-null  float64
 4   Cholesterol Level         56137 non-null  int64  
 5   Glucose Level             56

In [5]:
# Prepare training data
target_col = 'Cardiovascular Disease'
feature_cols = [col for col in train_data.columns if col != target_col and col != 'id']

X_train = train_data[feature_cols]
y_train = train_data[target_col]

print(f"Features: {feature_cols}")
print(f"\nX_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"\nClass distribution in training:")
print(y_train.value_counts())

Features: ['Age', 'Gender', 'Systolic Blood Pressure', 'Diastolic Blood Pressure', 'Cholesterol Level', 'Glucose Level', 'Smoking Status', 'Alcohol Intake', 'Physical Activity', 'BMI']

X_train shape: (56137, 10)
y_train shape: (56137,)

Class distribution in training:
Cardiovascular Disease
1    28269
0    27868
Name: count, dtype: int64


In [6]:
# Prepare test data
X_test = test_data[feature_cols]
y_test = test_data[target_col]

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")
print(f"\nClass distribution in test:")
print(y_test.value_counts())

X_test shape: (13652, 10)
y_test shape: (13652,)

Class distribution in test:
Cardiovascular Disease
0    6882
1    6770
Name: count, dtype: int64


In [ ]:
# Train LightGBM model
print("Training LightGBM model...\n")

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'random_state': 42
}

# Convert to numpy arrays to avoid NumPy 2.0 compatibility issue
train_dataset = lgb.Dataset(X_train.values, label=y_train.values)

model = lgb.train(
    lgb_params,
    train_dataset,
    num_boost_round=1000,
    valid_sets=[train_dataset],
    valid_names=['train']
)

print("Model training completed!")

Training LightGBM model...



ValueError: Unable to avoid copy while creating an array as requested.
If using `np.array(obj, copy=False)` replace it with `np.asarray(obj)` to allow a copy when needed (no behavior change in NumPy 1.x).
For more details, see https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword.

In [ ]:
# Make predictions on test set
y_pred_proba = model.predict(X_test, num_iteration=model.best_iteration)
y_pred = (y_pred_proba >= 0.5).astype(int)

print("Predictions completed!")

In [ ]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("\n=== Model Performance on Test Set ===")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC:   {roc_auc:.4f}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

print("\nConfusion Matrix:")
print(cm)

In [ ]:
# Classification Report
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['No CVD', 'CVD']))

In [ ]:
# Feature Importance
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance (Gain)')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance)

In [ ]:
# Save the model
model.save_model('saved_models_tausif/lightgbm_merged_dataset.txt')
print("\nModel saved to: saved_models_tausif/lightgbm_merged_dataset.txt")

In [ ]:
# Save metrics to file
metrics_summary = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC'],
    'Value': [accuracy, precision, recall, f1, roc_auc]
})

metrics_summary.to_csv('lightgbm_merged_metrics.csv', index=False)
print("\nMetrics saved to: lightgbm_merged_metrics.csv")
print(metrics_summary)